<a href="https://colab.research.google.com/github/yasyamauchi/education/blob/main/BME_AI_outliers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 人工知能 補助教材 (外れ値の検定)  
### 東洋大学 生命科学部/理工学部 生体医工学科

更新履歴：  
2026/8/17  
* 初版  

In [ ]:
# 1. 必要なライブラリのインストールとインポート
!pip install japanize-matplotlib -q

import matplotlib.pyplot as plt
import japanize_matplotlib
import numpy as np
import pandas as pd

# 再現性のための乱数シード
np.random.seed(42)

# 2. サンプルデータの生成（平均50、標準偏差10の正規分布データ 60件）
data = np.random.normal(loc=50, scale=10, size=60)

# 3. 第1四分位数 (Q1) と 第3四分位数 (Q3) の算出
q1 = np.percentile(data, 25)
q3 = np.percentile(data, 75)

# 条件に基づく外れ値の抽出（Q1未満 または Q3超過）
outliers = data[(data < q1) | (data > q3)]
normal_data = data[(data >= q1) & (data <= q3)]

# 4. 箱ひげ図の描画
fig, ax = plt.subplots(figsize=(8, 6))

# whis=0 に設定することで、ひげを伸ばさず Q1〜Q3 の外側すべてを外れ値マーカーとして描画
bp = ax.boxplot(
    data,
    whis=0,
    patch_artist=True,
    boxprops=dict(facecolor="skyblue", color="steelblue", alpha=0.7),
    medianprops=dict(color="darkblue", linewidth=2),
    flierprops=dict(
        marker="o",
        markerfacecolor="red",
        markeredgecolor="darkred",
        markersize=6,
        alpha=0.6,
    ),
)

# 四分位数の基準線を追加
ax.axhline(
    q3, color="darkorange", linestyle="--", label=f"第3四分位数 (Q3): {q3:.2f}"
)
ax.axhline(
    q1, color="forestgreen", linestyle="--", label=f"第1四分位数 (Q1): {q1:.2f}"
)

# グラフ装飾
ax.set_title("箱ひげ図（$Q_1$ 〜 $Q_3$ の区間外を外れ値とするデモ）", fontsize=14)
ax.set_ylabel("値", fontsize=12)
ax.set_xticklabels(["データセット"])
ax.legend(loc="upper right")
ax.grid(axis="y", linestyle=":", alpha=0.6)

plt.show()

# 5. 数値データの確認
print(f"総データ数: {len(data)}")
print(f"第1四分位数 (Q1): {q1:.2f}")
print(f"第3四分位数 (Q3): {q3:.2f}")
print(f"外れ値の件数: {len(outliers)} 件")
print(f"外れ値一覧:\n{np.sort(outliers)}")

In [ ]:
# 1. 必要なライブラリのインストールとインポート
!pip install japanize-matplotlib -q

import matplotlib.pyplot as plt
import japanize_matplotlib
import numpy as np
from scipy import stats

# 2. 前と同じサンプルデータ（平均50、標準偏差10の正規分布 60件）
np.random.seed(42)
data = np.random.normal(loc=50, scale=10, size=60)

# 3. 平均 (μ) と 標準偏差 (σ)、判定境界 (μ ± 2σ) の算出
mean = np.mean(data)
std = np.std(data, ddof=1)  # 不偏標準偏差

lower_bound = mean - 2 * std
upper_bound = mean + 2 * std

# 外れ値の判定 (|データ - 平均| >= 2σ)
is_outlier = (data < lower_bound) | (data > upper_bound)
normal_data = data[~is_outlier]
outliers = data[is_outlier]

# 4. グラフの描画（散布図と確率分布の2面構成）
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- 左図: 各データ点のプロット ---
indices = np.arange(len(data))
ax1.scatter(
    indices[~is_outlier],
    normal_data,
    color="steelblue",
    label="正常値 ($\\mu \\pm 2\\sigma$ 以内)",
    zorder=3,
)
ax1.scatter(
    indices[is_outlier],
    outliers,
    color="red",
    s=80,
    edgecolors="darkred",
    label="外れ値 ($|x - \\mu| \\geq 2\\sigma$)",
    zorder=4,
)

# 基準線と正常領域の帯
ax1.axhline(
    mean,
    color="darkblue",
    linestyle="-",
    linewidth=2,
    label=f"平均値 ($\\mu$): {mean:.2f}",
)
ax1.axhline(
    upper_bound,
    color="darkorange",
    linestyle="--",
    linewidth=1.5,
    label=f"上限 ($\\mu + 2\\sigma$): {upper_bound:.2f}",
)
ax1.axhline(
    lower_bound,
    color="forestgreen",
    linestyle="--",
    linewidth=1.5,
    label=f"下限 ($\\mu - 2\\sigma$): {lower_bound:.2f}",
)
ax1.axhspan(
    lower_bound,
    upper_bound,
    color="skyblue",
    alpha=0.2,
    label="正常範囲 (約95.4%)",
)

ax1.set_title("各データ点と $2\\sigma$ 基準による外れ値", fontsize=13)
ax1.set_xlabel("データインデックス (0〜59)", fontsize=11)
ax1.set_ylabel("値", fontsize=11)
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, linestyle=":", alpha=0.6)

# --- 右図: 理論分布（正規分布曲線）における 2σ 領域 ---
count, bins, _ = ax2.hist(
    data,
    bins=10,
    density=True,
    alpha=0.4,
    color="gray",
    edgecolor="black",
    label="サンプルヒストグラム",
)
x_axis = np.linspace(mean - 3.5 * std, mean + 3.5 * std, 200)
pdf = stats.norm.pdf(x_axis, mean, std)
ax2.plot(x_axis, pdf, color="black", linewidth=2, label="正規分布曲線")

# 領域の色分け
ax2.fill_between(
    x_axis,
    pdf,
    where=(x_axis < lower_bound) | (x_axis > upper_bound),
    color="red",
    alpha=0.3,
    label="外れ値領域 (外側 約4.6%)",
)
ax2.fill_between(
    x_axis,
    pdf,
    where=(x_axis >= lower_bound) & (x_axis <= upper_bound),
    color="skyblue",
    alpha=0.3,
    label="正常領域 (内側 約95.4%)",
)

ax2.axvline(mean, color="darkblue", linestyle="-", linewidth=2)
ax2.axvline(upper_bound, color="darkorange", linestyle="--", linewidth=1.5)
ax2.axvline(lower_bound, color="forestgreen", linestyle="--", linewidth=1.5)

ax2.set_title("正規分布における $2\\sigma$ ルールの理論領域", fontsize=13)
ax2.set_xlabel("値", fontsize=11)
ax2.set_ylabel("確率密度", fontsize=11)
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

# 5. 集計結果の確認
print(f"総データ数: {len(data)}")
print(f"平均値 (μ): {mean:.2f}")
print(f"標準偏差 (σ): {std:.2f}")
print(f"正常範囲 (μ ± 2σ): {lower_bound:.2f} 〜 {upper_bound:.2f}")
print(f"外れ値の件数: {len(outliers)} 件")
if len(outliers) > 0:
    print(f"外れ値一覧: {np.sort(outliers)}")